# Milvus + Llama-Index: Enhancing Deepseejk Assistant Agent with a Custom Retriever

这展示了如何通过Milvus定制的检索工具，增强基于Deepseek Assistant API构建的Llama-Index代理。

## Preparation

### 1. Install dependencies

In [1]:
import os

# NLTK 语料（stopwords / punkt_tab）存放在本地 model/nltk_data，避免联网下载被拦截
os.environ['NLTK_DATA'] = r'F:\Teewon\Milvue\model\nltk_data'

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
# os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

### 2. Start Milvus Service
There are 2 options to start a Milvus service:

- Zilliz Cloud: Zilliz provides cloud-native service for Milvus. It simplifies the process of deploying and scaling vector search applications by eliminating the need to create and maintain complex data infrastructure. Get Started Free!
- Open Source Milvus: You can install the open source Milvus using either Docker Compose or on Kubernetes.

Here, we use Milvus Lite to start with a lightweight version of Milvus, which works seamlessly with Google Colab and Jupyter Notebook.

In [2]:
from pymilvus import MilvusClient

mc=MilvusClient('../milvus_ingest.db')

### 3. Download example data

You can use any file(s) to build the knowledge base. We will use a SEC file uber_2021.pdf as an example.

In [3]:
!curl -L -o uber_2021.pdf https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/10k/uber_2021.pdf

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1836k  100 1836k    0     0  2399k      0 --:--:-- --:--:-- --:--:-- 2403k


## Getting Started
### 1. Set up Environment

你需要设置一些环境变量，例如传递你的 Deepseek API 密钥。请注意，你的 Deepseek 账户必须具备访问权限，并且拥有足够的配额以支持 Deepseek-v4-flash 模型。

In [4]:
import dotenv
dotenv.load_dotenv('../.env')

True

### 2. Customize Strategies

在此步骤中，我们将定义一些要使用的策略：

- 分块：配置文本分割器（例如 `chunk_size`）
- 嵌入：选择嵌入模型（例如 `BAAI/bge-small-en`）及其提供方（例如 HuggingFace、OpenAI）
- LLM：选择大语言模型（例如 `deepseek-v4-flash`），并设置模型参数（例如温度）。

In [5]:
from llama_index.core import Settings, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.deepseek import DeepSeek
from llama_index.vector_stores.milvus import MilvusVectorStore

llm=DeepSeek(
    model="deepseek-v4-flash",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
)

embed_model=HuggingFaceEmbedding(
    model_name='BAAI/bge-small-en',
    cache_folder=os.path.join(CUSTOM_CACHE, 'transformers'),
    device='cuda',
)

Settings.llm=llm
Settings.embed_model=embed_model
Settings.chunk_size=350

vector_store=MilvusVectorStore(
    uri='../milvus_ingest.db',
    dim=384,
    overwrite=True,
)
storage_context=StorageContext.from_defaults(vector_store=vector_store)

F:\Teewon\Milvue\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


### 3. Ingest Document(s)

In [6]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# Load document
docs=SimpleDirectoryReader(input_files=['./uber_2021.pdf']).load_data()

# Build index
vector_index=VectorStoreIndex.from_documents(docs,storage_context=storage_context)

[nltk_data] Error loading stopwords: Security Violation
[nltk_data]     [pathsec.urlopen]: SSRF attempt to restricted IP
[nltk_data]     198.18.0.36
[nltk_data] Error loading punkt_tab: Security Violation
[nltk_data]     [pathsec.urlopen]: SSRF attempt to restricted IP
[nltk_data]     198.18.0.36


### 4. Define Agent & Tool(s)

为了将向量存储索引与代理集成，我们需要将索引定义为一个检索工具。代理可以通过元数据中的工具名称和描述来识别该检索工具。

In [7]:
from llama_index.core.tools import RetrieverTool,ToolMetadata

milvus_tool=RetrieverTool(
    retriever=vector_index.as_retriever(similarity_top_k=3),
    metadata=ToolMetadata(
        name='CustomRetriever',
        description='Retrieve relevant information from provided documents.'
    )
)

接下来，我们来定义由 Deepseek Assistants API 驱动的智能体。创建智能体时，我们需要明确其角色、提供指令，并配备相应的工具。在这里，我们将让大语言模型的推理本身成为一位 SEC 分析师，并为其提供 Milvus 检索工具作为可用功能。

In [8]:
from llama_index.core.agent.workflow import AgentWorkflow

agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[milvus_tool],
    llm=llm,                                   # 就是上面定义的 DeepSeek 实例
    system_prompt="You are a QA assistant designed to analyze sec filings.",
    verbose=True,
)

## Try it out!

现在，该代理已准备好作为SEC分析师。它能够根据加载到Milvus中的文档来回应用户。

通过设置verbose=True，您可以了解代理回答问题时检索了哪些信息。

In [11]:
# AgentWorkflow 在 llama-index 0.14.x 用 run() 而非 chat()：
# run() 返回 WorkflowHandler（异步），必须 await 才能拿到结果。
# Jupyter (IPython 7+) 支持 cell 顶层 await。
response = await agent.run("What was Uber's revenue growth in 2021?")
print(response)


Running step init_run
Step init_run produced event AgentInput
Running step setup_agent
Step setup_agent produced event AgentSetup
Running step run_agent_step
Step run_agent_step produced event AgentOutput
Running step parse_agent_output
Step parse_agent_output produced no event
Running step call_tool
Step call_tool produced event ToolCallResult
Running step aggregate_tool_results
Step aggregate_tool_results produced event AgentInput
Running step setup_agent
Step setup_agent produced event AgentSetup
Running step run_agent_step
Step run_agent_step produced event AgentOutput
Running step parse_agent_output
Step parse_agent_output produced no event
Running step call_tool
Step call_tool produced event ToolCallResult
Running step aggregate_tool_results
Step aggregate_tool_results produced event AgentInput
Running step setup_agent
Step setup_agent produced event AgentSetup
Running step run_agent_step
Step run_agent_step produced event AgentOutput
Running step parse_agent_output
Step parse_ag